In [1]:
# Imports
import pandas as pd
import numpy as np
import joblib
from itertools import product
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split
import duckdb

In [2]:
# Define different options for features
# Base features: Game scenario + plate location
base = ["balls", "strikes", "outs_when_up", "p_throws", "stand", "on_1b", "on_2b", "on_3b", "plate_x", "z_diff", "swing"]

# Base + Pitch Metrics
base_w_pitch = base + ["release_speed", "release_pos_x", "release_pos_z", "pfx_x", "pfx_z", "release_spin_rate", "release_extension"]

# Define choices for hyperparameters and features (excluding learning rate, will define later)
md = [4, 6, 8, -1]
nl = [31, 63, 127, 255]
mcs = [100, 250, 500, 1000, 2000]
feats = {"base": base, "base_w_pitch": base_w_pitch}

In [3]:
# Define target variable
target = "delta_run_exp"

# Load Training Data

In [4]:
# Use DuckDB to read SQL queries
train_query = """
    SELECT
        *
    FROM
        data.parquet
    WHERE
        game_year < 2026
"""
df = duckdb.sql(train_query).df()

In [5]:
len(df)

3831575

In [6]:
df1 = df.dropna(subset=base_w_pitch)

In [7]:
len(df1)

3815918

# Create Function To Build Models

In [8]:
# Function that builds model based on iteration of hyperparamters
# Evaluates with validation set, which optimizes for MSE
def build_model(train_df, val_df, features, target, learning_rate, max_depth, num_leaves, min_child_samples):
    
    model = LGBMRegressor(
        n_estimators=10000,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
    
    model.fit(
        train_df[features], train_df[target],
        eval_set=[(val_df[features], val_df[target])],
        eval_metric="l2",
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )
    
    return model, model.best_iteration_, model.best_score_["valid_0"]["l2"]

In [9]:
# Create training and validation set
# 90% of data for training, 10% for validation
train_df, val_df = train_test_split(df1, test_size=0.1, random_state=42)

In [10]:
# Grid - all combinations of defined features/parameters
# Ensures num_leaves and max_depth are within reasonable correlation
grid = [
    (fname, d, l, m)
    for fname, d, l, m in product(feats, md, nl, mcs)
    if d == -1 or l <= 0.65 * 2**d
]

# Stage 1

#### Using a set learning rate (0.1), find the best performing combinations of features and parameters

In [11]:
stage1 = []
# Loop through each feature/parameters combo
for fname, d, l, m in grid:

    # Attain optimal # of boosting rounds and MSE score from this iteration
    _, n_trees, score = build_model(train_df, val_df, feats[fname], target, 0.1, d, l, m)

    # Create row with parameters and score
    stage1.append({"features": fname, 
                   "learning_rate": 0.1, 
                   "max_depth": d, 
                   "num_leaves": l,
                   "min_child_samples": m, 
                   "n_trees": n_trees, 
                   "val_mse": score})

    # Save as we go
    pd.DataFrame(stage1).to_csv("tuning_stage1.csv", index=False)

# Create df of all iterations
stage1_df = pd.DataFrame(stage1).sort_values("val_mse")

In [12]:
stage1_df

,features,learning_rate,max_depth,num_leaves,min_child_samples,n_trees,val_mse
58,base_w_pitch,0.1,8,127,1000,181,0.045348
57,base_w_pitch,0.1,8,127,500,171,0.045352
56,base_w_pitch,0.1,8,127,250,160,0.045354
52,base_w_pitch,0.1,8,63,500,330,0.045357
55,base_w_pitch,0.1,8,127,100,170,0.045358
...,...,...,...,...,...,...,...
3,base,0.1,6,31,1000,291,0.045460
4,base,0.1,6,31,2000,396,0.045463
38,base,0.1,-1,255,1000,50,0.045464
35,base,0.1,-1,255,100,45,0.045466


# Stage 2

#### Try the best parameters with lower learning rates to see if mse scores improve

In [13]:
# 5 Best performing parameters
top = stage1_df.head(5)

stage2 = []
# Top 5 parameter combos
for _, row in top.iterrows():
    
    # New learning rates to try
    for lr in [0.01, 0.02, 0.05]:

        # Attain optimal # of boosting rounds and MSE score from this iteration
        _, n_trees, score = build_model(train_df, val_df, feats[row["features"]], target, lr, int(row["max_depth"]), int(row["num_leaves"]), int(row["min_child_samples"])
        )

        # Create row with parameters and score
        stage2.append({"features": row["features"], 
                       "learning_rate": lr,
                       "max_depth": int(row["max_depth"]), 
                       "num_leaves": int(row["num_leaves"]),
                       "min_child_samples": int(row["min_child_samples"]),
                       "n_trees": n_trees, "val_mse": score})

        # Save as we go
        pd.DataFrame(stage2).to_csv("tuning_stage2.csv", index=False)

In [14]:
# Combine results
all_results = pd.concat([stage1_df, pd.DataFrame(stage2)]).sort_values("val_mse")

In [15]:
all_results

,features,learning_rate,max_depth,num_leaves,min_child_samples,n_trees,val_mse
7,base_w_pitch,0.02,8,127,250,886,0.045344
14,base_w_pitch,0.05,8,127,100,317,0.045346
3,base_w_pitch,0.01,8,127,500,1532,0.045346
58,base_w_pitch,0.10,8,127,1000,181,0.045348
4,base_w_pitch,0.02,8,127,500,690,0.045349
...,...,...,...,...,...,...,...
3,base,0.10,6,31,1000,291,0.045460
4,base,0.10,6,31,2000,396,0.045463
38,base,0.10,-1,255,1000,50,0.045464
35,base,0.10,-1,255,100,45,0.045466


In [16]:
# Best performing parameters
best = all_results.iloc[0]
print(best)

features             base_w_pitch
learning_rate                0.02
max_depth                       8
num_leaves                    127
min_child_samples             250
n_trees                       886
val_mse                  0.045344
Name: 7, dtype: object


# Final Model

In [17]:
# Use parameters from best fit to create final model
final_model = LGBMRegressor(
    n_estimators=int(best["n_trees"] * 1.1),   # slightly more trees for ~11% more data
    learning_rate=best["learning_rate"],
    max_depth=int(best["max_depth"]),
    num_leaves=int(best["num_leaves"]),
    min_child_samples=int(best["min_child_samples"]),
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

# Fit final model
final_model.fit(df1[base_w_pitch], df1[target])

# Save model
joblib.dump(final_model, "swing_decision_model.pkl")

['swing_decision_model.pkl']